In [1]:
!pip install -q kaggle

In [2]:
from google.colab import userdata
import os

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

In [3]:
api.dataset_download_files("ankurzing/sentiment-analysis-for-financial-news", path="./", unzip=True)

Dataset URL: https://www.kaggle.com/datasets/ankurzing/sentiment-analysis-for-financial-news


In [4]:
import pandas as pd

In [5]:
df = pd.read_csv('all-data.csv', header=None, encoding='latin')
df.columns = ['sentiment', 'text']

In [6]:
import random
import torch
import numpy as np

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [7]:
from bs4 import BeautifulSoup
# beautifulsoap is for html parsing
import re
# re is regex and regex is for searching, matching, extracting, replacing, and validating text based on patterns.
import nltk
from nltk.corpus import stopwords, wordnet
#we first import the stopwords modules, which knows how to access the data sets
nltk.download('stopwords')
#then we are downloading the stopwords dataset from NLTK servers to our local database
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
def text_preprocessing(text):
  text = BeautifulSoup(text,"html.parser").get_text()   #removes HTML tags
  text = re.sub(r"https\S+|www\S+",'',text)
  #removes/subtracts URLs untill there is a whitespace character (S+ represnts to subtract all the following characters after https until there is non-white space)
  text = re.sub(r"[^a-zA-Z0-9\s]",'',text)
  #removes charcters other than alphabets and digits
  text = text.lower()
  #converting to lowercase
  text = ' '.join(word for word in text.split() if word not in stop_words)
  return text

In [9]:
from sklearn.preprocessing import LabelEncoder
# when are going into machine learning model so we should use scikit learn modules
encoder = LabelEncoder()
df['sentiment'] = encoder.fit_transform(df['sentiment'])

In [10]:
from sklearn.model_selection import train_test_split
train_df,temp_df = train_test_split(df,test_size = 0.3, random_state = 42, stratify = df['sentiment'], shuffle = True)
# stratify makes sure that the ratios of each type of sentiment in the train_df and temp_df remains same
#shuffle is for shuffling the dataset before splitting and random seed(42) applies only during the split

In [11]:
val_df,test_df = train_test_split(temp_df,test_size = 0.5, random_state = 42, stratify = temp_df['sentiment'], shuffle = True)

In [12]:
import nltk
from nltk.corpus import wordnet, stopwords

nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [13]:
def get_synonyms(word):
    synonyms = set()

    for synset in wordnet.synsets(word):
        for lemma in synset.lemmas():
            synonym = lemma.name().replace("_", " ").lower()

            if synonym != word.lower() and " " not in synonym:
                synonyms.add(synonym)

    return list(synonyms)

In [14]:
def synonym_replacement(text, n=1):
    words = text.split()
    new_words = words.copy()

    # Do not replace stopwords, numbers, or very short tokens
    candidates = [word for word in words if word.lower() not in stop_words and word.isalpha() and len(word) > 2]

    np.random.shuffle(candidates)

    replacements = 0

    for word in candidates:
        synonyms = get_synonyms(word.lower())

        if synonyms:
            replacement = np.random.choice(synonyms)

            # Preserve simple capitalisation
            if word[0].isupper():
                replacement = replacement.capitalize()

            new_words = [replacement if current_word == word else current_word for current_word in new_words]

            replacements += 1

        if replacements >= n:
            break

    return " ".join(new_words)

In [15]:
# Augment only minority classes in the training data to mitigate the baisness of the dataset
negative_aug = train_df[train_df["sentiment"] == 0].copy()
positive_aug = train_df[train_df["sentiment"] == 2].copy()

# Augmentation is only done in training and following are the major reasons:
# 1) Increasing in training can give more varied examples and if done to validation set , usually validation set is the one which decides the best
   # epoch so there is a chance of validation becoming less meaningful when adding extra sentences(some times the meanings may change) and cannot
   # obviously add to the test dataset because to check the accuracy of the model we need to test on real data.

# 2) Data Leakage: If we add to the full data there is a change of original sentence in the test dataset and the fabricated in the validation set
   # which is a data leakage.

negative_aug["text"] = negative_aug["text"].apply(
    lambda x: synonym_replacement(x, n=1)
)

positive_aug["text"] = positive_aug["text"].apply(
    lambda x: synonym_replacement(x, n=1)
)

train_df = pd.concat(
    [train_df, negative_aug, positive_aug],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

In [26]:
train_df

,sentiment,text
0,0,SRV lowered its net sales estimate for the who...
1,2,The airline has ordered nine Airbus A350-900 a...
2,2,Luxembourg-registered investment fund Amber Tr...
3,2,NASDAQ-listed Yahoo Inc has premise a new serv...
4,1,Finnish consumers prefer to buy the cheapest b...
...,...,...
4764,2,Metso expects its net sales to increase by abo...
4765,2,Stora Enso 's third-quarter pre-tax profit dou...
4766,1,The company has an annual turnover of EUR32 .8 m.
4767,0,"( ADP News ) - Jan 27 , 2009 - Suomi industria..."


In [27]:
train_df['sentiment'].value_counts()

,count
sentiment,
1,2015
2,1908
0,846


Downloading the Pre Trained Tokenizer & Model

Teacher Model: FinBERT Model

In [16]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
#Pretrained Teacher Model & Tokenizer
import torch


teacher_model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(teacher_model_name)
#This line downloads the tokens of the pretrained teacher model

teacher_model = AutoModelForSequenceClassification.from_pretrained(teacher_model_name)
#This loads the FinBERT model/complete architecture

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Determining device for the model

teacher_model.to(device)
# Moving model to GPU (if available)

teacher_model.eval()
# Teacher model will only be used for inference/evaluation

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [17]:
from transformers import AutoModel

student_model_name = "distilbert-base-uncased"
#This model has only 6 transformer layer whereas teacher model has 12 transformer layers
#and here uncased means this model unclases every alphabet into small letters.

student_model = AutoModel.from_pretrained(student_model_name)
#This only loads the encoder part and after this I am going to define the customBERT model
#which places our own dense layers to make the sentiment prediction.

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
X_train = tokenizer(train_df['text'].tolist(),max_length=128,padding=True,truncation=True,return_tensors="pt")
#Converts the words into token IDs

X_val = tokenizer(val_df['text'].tolist(),max_length=128,padding=True,truncation=True,return_tensors="pt")
#Converts the words into token IDs

y_train = train_df['sentiment'].values
y_val = val_df['sentiment'].values
#Converts the sentiment types into values of 0 --> negative ,1 --> neutral. ,2 --> positive

In [19]:
# Following is the custom BERT model which will be used on top of pretrained and imported student model
import torch
import torch.nn as nn

class CustomBERTModel(nn.Module):

    def __init__(self, student_model):
        super(CustomBERTModel, self).__init__()

        self.bert = student_model
        # DistilBERT encoder

        self.norm = nn.LayerNorm(768)
        # Layer Normalization

        # Fully Connected Layers
        self.dense1 = nn.Linear(768, 256)
        self.dropout1 = nn.Dropout(0.3)

        self.dense2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(0.2)

        self.dense3 = nn.Linear(128, 64)
        self.dropout3 = nn.Dropout(0.1)

        self.output_layer = nn.Linear(64, 3)

        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask):

        outputs = self.bert(input_ids=input_ids,attention_mask=attention_mask)
        # DistilBERT output is the embeddings that are generated for the provided input

        x = outputs.last_hidden_state

        x = self.norm(x)
         # Layer Normalization

        mask = attention_mask.unsqueeze(-1).float()
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

        # Dense Layers
        x = self.relu(self.dense1(x))
        x = self.dropout1(x)

        x = self.relu(self.dense2(x))
        x = self.dropout2(x)

        x = self.relu(self.dense3(x))
        x = self.dropout3(x)

        logits = self.output_layer(x)

        return logits

In [20]:
# Initializing CustomBERTModel with the base DistilBertModel
student_model = CustomBERTModel(student_model)
student_model.to(device)

CustomBERTModel(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear

In [21]:
import torch.nn.functional as F

def distillation_loss(student_logits,teacher_logits,labels,alpha=0.5,temperature=3.0):

    student_soft = F.log_softmax(student_logits / temperature, dim=1)
    teacher_soft = F.softmax(teacher_logits / temperature, dim=1)
    # student and teacher soft targets

    kl_loss = F.kl_div(student_soft,teacher_soft,reduction='batchmean') * (temperature ** 2)
    # KL Divergence Loss

    ce_loss = F.cross_entropy(student_logits,labels)
    # Hard Label Loss

    loss = alpha * kl_loss + (1 - alpha) * ce_loss
    # Combined Loss

    return loss

In [22]:
import torch.optim as optim

optimizer = optim.Adam(student_model.parameters(),lr=5e-5)
#importing adam optmizer

In [23]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train["input_ids"],X_train["attention_mask"],torch.tensor(y_train, dtype=torch.long))
#this step packages the tokenized data to feed the model

val_dataset = TensorDataset(X_val["input_ids"],X_val["attention_mask"],torch.tensor(y_val, dtype=torch.long))

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
#this step packs the data into batches of 32 sizes and suffles the data after each epoch

val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False)

In [24]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
from copy import deepcopy
import torch.optim as optim

epochs = 12

# Define optimizer before scheduler
optimizer = optim.Adam(student_model.parameters(),lr=5e-5)
scheduler = ReduceLROnPlateau(optimizer,mode="min",factor=0.5,patience=2)

best_val_acc = 0
best_weights = None

patience = 5
patience_counter = 0

history = { "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [] }

for epoch in range(epochs):

    # Training

    student_model.train()

    running_loss = 0
    correct = 0
    total = 0

    for input_ids, attention_mask, labels in train_loader:

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.no_grad():
             teacher_logits = teacher_model(input_ids=input_ids,attention_mask=attention_mask).logits
             # insdie this embeddings are generated and then converted and finally we get teacher logits

             teacher_logits = teacher_logits[:, [1, 2, 0]]

        student_logits = student_model( input_ids, attention_mask)
        #This produces the student logits

        # print(student_logits[:5])

        loss = distillation_loss( student_logits, teacher_logits, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predictions = student_logits.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total


    # Validation

    student_model.eval()
    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for input_ids, attention_mask, labels in val_loader:

            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            teacher_logits = teacher_model(input_ids=input_ids, attention_mask=attention_mask).logits

            teacher_logits = teacher_logits[:, [1, 2, 0]]

            student_logits = student_model( input_ids, attention_mask )

            loss = distillation_loss( student_logits, teacher_logits, labels)

            running_loss += loss.item()

            predictions = student_logits.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    val_loss = running_loss / len(val_loader)
    val_acc = correct / total

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc
        best_weights = deepcopy(student_model.state_dict())

        torch.save(best_weights,"distilled_model.pth")

        patience_counter = 0

    else:

        patience_counter += 1

        if patience_counter >= patience:

            print("Early stopping triggered.")
            break

if best_weights is not None:
    student_model.load_state_dict(best_weights)

Epoch [1/12] Train Loss: 0.7778 Train Acc: 0.6777 Val Loss: 0.3792 Val Acc: 0.7868
Epoch [2/12] Train Loss: 0.2902 Train Acc: 0.8805 Val Loss: 0.4030 Val Acc: 0.8267
Epoch [3/12] Train Loss: 0.2246 Train Acc: 0.9216 Val Loss: 0.3051 Val Acc: 0.8446
Epoch [4/12] Train Loss: 0.1829 Train Acc: 0.9467 Val Loss: 0.2884 Val Acc: 0.8432
Epoch [5/12] Train Loss: 0.1641 Train Acc: 0.9553 Val Loss: 0.2975 Val Acc: 0.8349
Epoch [6/12] Train Loss: 0.1516 Train Acc: 0.9614 Val Loss: 0.2772 Val Acc: 0.8391
Epoch [7/12] Train Loss: 0.1446 Train Acc: 0.9639 Val Loss: 0.2693 Val Acc: 0.8404
Epoch [8/12] Train Loss: 0.1409 Train Acc: 0.9644 Val Loss: 0.2712 Val Acc: 0.8418
Early stopping triggered.


In [25]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_test = tokenizer( test_df["text"].tolist(), max_length=128, padding=True, truncation=True, return_tensors="pt")

test_dataset = TensorDataset( X_test["input_ids"], X_test["attention_mask"], torch.tensor(test_df["sentiment"].values, dtype=torch.long))

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

student_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for input_ids, attention_mask, labels in test_loader:
        logits = student_model( input_ids.to(device), attention_mask.to(device))
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())

print("Test accuracy:", accuracy_score(all_labels, all_preds))
print(classification_report( all_labels, all_preds, target_names=encoder.classes_))
print(confusion_matrix(all_labels, all_preds))

Test accuracy: 0.844566712517194
              precision    recall  f1-score   support

    negative       0.77      0.91      0.83        91
     neutral       0.88      0.88      0.88       432
    positive       0.82      0.75      0.78       204

    accuracy                           0.84       727
   macro avg       0.82      0.85      0.83       727
weighted avg       0.85      0.84      0.84       727

[[ 83   7   1]
 [ 21 378  33]
 [  4  47 153]]
